# RIFT — первые 200 файлов в Google Drive

**Без VPS, SSH, Google Cloud Console и передачи OAuth-токенов.**

Этот блокнот использует Python-код из вашего публичного GitHub-репозитория `UsernameMP/English`, ветка `rift-source-index`, и Google Colab. Колаб откроет диалог доступа к Google Drive: **проверьте выбранный аккаунт перед разрешением**. Разрешение даёт исполняемому коду доступ к содержимому Диска, поэтому запускайте этот блокнот только из своего репозитория и после проверки исходного кода.

### Что нажать

1. В Colab выберите **Среда выполнения → Выполнить все** (Runtime → Run all).
2. При запросе подтвердите доступ Colab к своему Google Drive.
3. Результаты появятся в **Мой диск → RIFT**: `raw/` — неизменённые PDF/документы/архивы, `receipts/` — ссылки и контрольные суммы, `index/` — резервные копии SQLite, `reports/` — статистика.

Выборка: 5 000 URL из **1 240 доменов**, первым проходом по одному домену. Цель — **ровно 200 разных сохранённых файлов**, не 200 проверенных URL. Сайты опрашиваются последовательно, с паузой минимум 10 секунд на домен и соблюдением robots.txt. HTML и картинки пока пропускаются.

**Важно:** Google Colab может прервать сеанс. Чтобы продолжить, откройте этот же блокнот и снова выполните все ячейки: программа восстановит прогресс по сохранённым индексам и квитанциям файлов. SQLite работает только на временном локальном диске Colab; резервные копии записываются на Google Drive. Drive может временно ограничивать число файловых операций. Запись не подтверждается API Google Drive: выполняется контрольное чтение через подключённый диск; окончательную синхронизацию можно отдельно проверить в веб-интерфейсе Drive.

Никакие пароли, токены Google или содержимое документов не публикуются в GitHub. Система не изменяет ваши существующие файлы вне папки RIFT. Не используйте этот механизм для обхода запретов сайтов или для дальнейшей публикации чужих материалов без соответствующих прав.


In [ ]:
# @title 1. Установить проверенный код RIFT из GitHub
import subprocess, sys
from pathlib import Path
REPO = Path('/content/rift_source')
if not (REPO/'rift_downloader'/'mounted_drive.py').is_file():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',
                    '--branch','rift-source-index',
                    'https://github.com/UsernameMP/English.git',str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'sparse-checkout','set',
                    'rift_downloader','source_discovery/resources_shards'],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','requests>=2.31,<3'],check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0,str(REPO))
from rift_downloader.db import connect, completed, exclusive_runner
from rift_downloader.cli import records,run_pilot
from rift_downloader.pilot_sample import seed_pilot
from rift_downloader.mounted_drive import (MountedDriveUploader,restore_index_snapshot,reconcile_receipts)
from rift_downloader.backup import backup_index,last_backup_count
from rift_downloader.report import export_report
print('Код RIFT установлен. Документы ещё не скачивались.')


In [ ]:
# @title 2. Подключить ваш Google Drive (потребуется нажать «Разрешить»)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RIFT')
if not DRIVE_ROOT.parent.is_dir():
    raise RuntimeError('Не найдена папка MyDrive. Проверьте, какой Google-аккаунт подключён.')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
print('Архив будет сохранён в папку:',DRIVE_ROOT)


In [ ]:
# @title 3. Скачать первые 200 оригинальных файлов и сохранить отчёт
import json, shutil, traceback
from types import SimpleNamespace

WORK=Path('/content/rift_pilot_state')
WORK.mkdir(parents=True,exist_ok=True)
DB=WORK/'rift.sqlite3'
SPOOL=WORK/'spool'
SPOOL.mkdir(parents=True,exist_ok=True)
RESTORE=restore_index_snapshot(DRIVE_ROOT,DB)
print('Восстановление индекса:',RESTORE)
db=connect(DB)
count=db.execute('SELECT COUNT(*) FROM resources').fetchone()[0]
if count==0:
    patterns=[str(REPO/'source_discovery'/'resources_shards'/'resource_*.jsonl')]
    imported=__import__('rift_downloader.db',fromlist=['import_resources']).import_resources(db,records(patterns))
    if imported<1000:
        raise RuntimeError('Индекс RIFT импортирован неполностью: '+str(imported))
    print('Импортировано URL:',imported)
else:
    print('Найден сохранённый индекс URL:',count)
RECOVERY=reconcile_receipts(db,DRIVE_ROOT)
print('Восстановлено по квитанциям файлов:',RECOVERY)
PLAN=seed_pilot(db,limit=5000)
print('План пилота:',PLAN,'/ уже проверенных объектов:',completed(db))
uploader=MountedDriveUploader(DRIVE_ROOT)
args=SimpleNamespace(
    limit_success=200,max_checks=3000,
    min_delay=10.0,max_file_mb=50,spool_max_gb=5,max_retries=4,
    backup_every=25,seed_limit=5000,spool=str(SPOOL))
stop_reason='not_started'
checkpoint=None
try:
    with exclusive_runner(DB):
        if completed(db)>=args.limit_success:
            stop_reason='already_completed'
        else:
            stop_reason=run_pilot(db,args,uploader)
finally:
    # Snapshot the current index even on interruption / quota errors.
    try:
        if completed(db)>last_backup_count(db):
            checkpoint=backup_index(db,uploader,SPOOL)
    except Exception as exc:
        print('Резервная копия индекса не записалась:',type(exc).__name__,str(exc)[:180])
    try:
        paths=export_report(db,WORK/'pilot_report.json')
        report_dir=DRIVE_ROOT/'reports'
        report_dir.mkdir(parents=True,exist_ok=True)
        for value in paths.values():
            source=Path(value)
            shutil.copy2(source,report_dir/source.name)
        print('Отчёты скопированы в RIFT/reports')
    except Exception as exc:
        print('Ошибка экспорта отчёта:',type(exc).__name__,str(exc)[:180])
print('Причина остановки:',stop_reason)
print('Уникальных сохранённых файлов:',completed(db),'из',args.limit_success)
print('Последняя резервная копия:',checkpoint)
if completed(db)<args.limit_success:
    print('Если сеанс прервался или очередь повторных попыток отложена, запустите все ячейки ещё раз.')
